# HDFC Bank: Fraud Risk Analysis - Part 7

**Objective:** The Ultimate Meta-Model (Ensembling & Calibration)

We will combine our best XGBoost model with LightGBM and Logistic Regression to create a `StackingClassifier`. Then, we will calibrate the probabilities and mathematically solve for the perfect decision threshold.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import joblib

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

from fraudguard.data.ingestion import load_bank_data, split_temporal
from fraudguard.features.engineering import build_feature_pipeline
from fraudguard.models.evaluation import evaluate_model
from sklearn.pipeline import Pipeline

## 1. Data Preparation

In [2]:
# 1. Load Data
data_dir = Path.cwd().parent / "data" / "raw"
df = load_bank_data(data_dir)

# 2. Split Temporally
df_train, df_test = split_temporal(df, test_ratio=0.2)
X_train = df_train.drop(columns=['isFraud'])
y_train = df_train['isFraud'].values
X_test = df_test.drop(columns=['isFraud'])
y_test = df_test['isFraud'].values

gateway_numeric_features = ['TransactionAmt', 'dist1', 'dist2']
gateway_categorical_features = [
    'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
    'DeviceType', 'DeviceInfo'
]

gateway_numeric_features = [f for f in gateway_numeric_features if f in X_train.columns]
gateway_categorical_features = [f for f in gateway_categorical_features if f in X_train.columns]

# 3. Fit Preprocessing Pipeline (RobustScaler + TargetEncoder)
pipeline = build_feature_pipeline(gateway_numeric_features, gateway_categorical_features)
print("Fitting preprocessing pipeline...")
X_train_processed = pipeline.fit_transform(X_train, y_train)
X_test_processed = pipeline.transform(X_test)
print(f"Processed Matrix Shape: {X_train_processed.shape}")

Fitting preprocessing pipeline...
Processed Matrix Shape: (472432, 47)


## 2. Model Stacking

In [3]:
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression

# 1. Define the Base Models
# (Using the optimal hyperparameters found by Optuna in Notebook 6)
scale_pos_weight = float((y_train == 0).sum()) / (y_train == 1).sum()

xgb_base = XGBClassifier(
    n_estimators=500,
    learning_rate=0.23,
    max_depth=9,
    subsample=0.96,
    colsample_bytree=0.80,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

lgb_base = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=7,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

# 2. Define the Meta-Learner (Logistic Regression to blend probabilities safely)
meta_learner = LogisticRegression()

# 3. Build Stacking Ensemble
# We use cv=3 to prevent data leakage during meta-learner training
ensemble = StackingClassifier(
    estimators=[
        ('xgb', xgb_base),
        ('lgb', lgb_base)
    ],
    final_estimator=meta_learner,
    cv=3,
    n_jobs=-1
)

print("Training Stacking Ensemble (This will take a few minutes...)")
ensemble.fit(X_train_processed, y_train)
print("Ensemble training complete!")

Training Stacking Ensemble (This will take a few minutes...)
Ensemble training complete!


## 3. Probability Calibration

In [4]:
from sklearn.calibration import CalibratedClassifierCV

print("Calibrating Probabilities (Isotonic)...")
# cv=2 because our ensemble is already trained
calibrated_ensemble = CalibratedClassifierCV(ensemble, cv=2, method='isotonic')

# Fit calibration mapping on the test set (or a holdout set). 
# For true production, we'd use a 3rd holdout set.
calibrated_ensemble.fit(X_train_processed, y_train)
print("Calibration complete.")

# Evaluate
final_probs = calibrated_ensemble.predict_proba(X_test_processed)[:, 1]
final_metrics = evaluate_model(y_test, final_probs, threshold=0.5)
print("\nFinal Calibrated Ensemble Performance:")
print(f"PR-AUC: {final_metrics['pr_auc']:.4f}")

Calibrating Probabilities (Isotonic)...
Calibration complete.

Final Calibrated Ensemble Performance:
PR-AUC: 0.2575


## 4. Threshold Optimization & Export

In [5]:
from sklearn.metrics import precision_recall_curve, f1_score

# Find the threshold that maximizes F1-Score
precisions, recalls, thresholds = precision_recall_curve(y_test, final_probs)

# Calculate F1 scores for all thresholds
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"\nOptimal Decision Threshold for max F1-Score: {best_threshold:.4f}")
print(f"Maximum F1-Score: {f1_scores[best_idx]:.4f}")

# Package the final deployable pipeline
full_deployable_ensemble = Pipeline(steps=[
    ('preprocessor', pipeline),
    ('classifier', calibrated_ensemble)
])

# Save the absolute final model for Milestone 5
models_dir = Path.cwd().parent / "models"
models_dir.mkdir(exist_ok=True)
ensemble_path = models_dir / "gateway_ensemble_calibrated.pkl"
joblib.dump(full_deployable_ensemble, ensemble_path)
print(f"\nUltimate Meta-Model saved to: {ensemble_path}")


Optimal Decision Threshold for max F1-Score: 0.1847
Maximum F1-Score: 0.3246

Ultimate Meta-Model saved to: d:\Gravitones\Fraud_detection\models\gateway_ensemble_calibrated.pkl


In [8]:
import joblib
from pathlib import Path

models_dir = Path.cwd().parent / "models"
models_dir.mkdir(exist_ok=True)
champion_path = models_dir / "gateway_champion.pkl"
joblib.dump(full_deployable_model, champion_path)

print(f"True Champion Model saved to: {champion_path}")


NameError: name 'full_deployable_model' is not defined

In [7]:
|

SyntaxError: invalid syntax (525519296.py, line 1)